In [ ]:
# import necessary libs and util functions and save intermediate results for validation and the final dataframe with the mask 
# and reflectance data and also numpy files for mask


In [ ]:
# 1. Learn what are .bin and .hdr file formats and how they store the data ot he reflectance data
# 2. Extract the reflectance data from these fiels

### Extract Reflectance

In [12]:
%matplotlib qt

In [3]:
from skimage.morphology import remove_small_objects, remove_small_holes, opening, disk
import spectral.io.envi as envi
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
import numpy as np
import os

In [4]:

data_root = r'C:\Users\baolab\Downloads\biochar_data'

sub_folders = os.listdir(data_root)

print(sub_folders)

['BH7030', 'BH8020', 'BH9010', 'control']


In [5]:
i = 0
data = f'C:/Users/baolab/Downloads/biochar_data\{sub_folders[i]}'

files1 = os.listdir(data)
print(set(files1))
print(len(files1))

file_names_list = []

for i in range(len(files1)):
    
    file_name_1 = files1[i].split('.')[0]
    file_names_list.append(file_name_1)

file_names_list = list(set(file_names_list))

print(file_names_list)
print(len(file_names_list))


{'7030_Oven dried_39_112_2026-02-24_13-01-53_.bin', '7030_Oven dried_1_1_2026-02-24_11-07-50_.bin', '7030_Oven dried_39_112_2026-02-24_13-01-53_.hdr', '7030_Oven dried_34_99_2026-02-24_12-52-25_.hdr', '7030_Oven dried_1_1_2026-02-24_11-07-50_.png', '7030_Oven dried_39_112_2026-02-24_13-01-53_.png', '7030_Oven dried_1_1_2026-02-24_11-07-50_.hdr', '7030_Oven dried_34_99_2026-02-24_12-52-25_.png', '7030_Oven dried_34_99_2026-02-24_12-52-25_.bin'}
9
['7030_Oven dried_39_112_2026-02-24_13-01-53_', '7030_Oven dried_34_99_2026-02-24_12-52-25_', '7030_Oven dried_1_1_2026-02-24_11-07-50_']
3


In [6]:
file_names_list

['7030_Oven dried_39_112_2026-02-24_13-01-53_',
 '7030_Oven dried_34_99_2026-02-24_12-52-25_',
 '7030_Oven dried_1_1_2026-02-24_11-07-50_']

In [7]:

def load_cube(file_name, data_path):
    '''
    Takes in the filename and return its cube
    '''
    data_dir = Path(data_path)

    header_file = str(data_dir / f'{file_name}.hdr')
    spectral_file = str(data_dir /f'{file_name}.bin')

    # open hyperspec cude
    image_cube = envi.open(header_file, spectral_file)

    print(f"Image dimensions (rows, cols, bands): {image_cube.shape}")
    print(f"Number of wavelengths (bands): {image_cube.nbands}")

    return image_cube


In [8]:
data

'C:/Users/baolab/Downloads/biochar_data\\BH7030'

In [9]:
file_name = file_names_list[0]
data_path = data

img_cube1 = load_cube(file_name, data_path)
img_cube1

Image dimensions (rows, cols, bands): (720, 640, 201)
Number of wavelengths (bands): 201


	Data Source:   'C:\Users\baolab\Downloads\biochar_data\BH7030\7030_Oven dried_39_112_2026-02-24_13-01-53_.bin'
	# Rows:            720
	# Samples:         640
	# Bands:           201
	Interleave:        BIP
	Quantization:  32 bits
	Data format:   float32

In [10]:
img_cube1.shape

(720, 640, 201)

In [13]:
band_index = 50

single_band = img_cube1.read_band(band_index)

import matplotlib.pyplot as plt

plt.figure(figsize=(8, 8))

plt.imshow(single_band)#, cmap='gray' # FOR GRAY SCALE IMAGE

plt.show()

In [14]:
# %matplotlib widget

def get_plant_px(img_cube):

    hypercube = img_cube.load()

    display_band = hypercube[:, :, 50]

    plant_coords = [] # List to store plant coordinates

    fig, ax = plt.subplots()

    ax.imshow(display_band, cmap='viridis')
    ax.set_title("Click to select PLANT pixels")

    def onclick_plant(event):

        # Get integer coordinates of the click
        x, y = int(event.xdata), int(event.ydata)

        # Store the coordinates (row, col)
        plant_coords.append([y, x])

        # Draw a marker for visual feedback
        ax.plot(x, y, 'rx', markersize=8)
        fig.canvas.draw()

    # Connect the click event to our function
    cid = fig.canvas.mpl_connect('button_press_event', onclick_plant)
    plt.show() # Display the interactive plot

    return plant_coords

In [15]:
plantsp = get_plant_px(img_cube1)

c:\ProgramData\anaconda3\envs\hyperspec\Lib\site-packages\spectral\io\spyfile.py:224: NaNValueWarning: Image data contains NaN values.
  warnings.warn('Image data contains NaN values.', NaNValueWarning)


In [18]:
print(plantsp)

[[387, 271], [365, 272], [350, 272], [337, 273], [331, 285], [328, 302], [328, 310], [324, 321], [322, 330], [322, 338], [336, 337], [344, 337], [368, 338], [380, 331], [380, 341], [392, 329], [392, 329], [387, 313], [387, 299], [391, 293], [376, 261], [357, 261], [357, 288], [348, 291], [344, 304], [341, 320], [331, 327], [350, 327], [363, 326], [365, 322], [365, 321], [379, 295], [375, 281], [375, 276], [393, 282], [388, 263], [393, 267], [365, 304], [359, 304], [351, 316], [372, 316], [372, 306], [366, 288], [357, 341]]


In [17]:
print(plantsp)

[[387, 271], [365, 272], [350, 272], [337, 273], [331, 285], [328, 302], [328, 310], [324, 321], [322, 330], [322, 338], [336, 337], [344, 337], [368, 338], [380, 331], [380, 341], [392, 329], [392, 329], [387, 313], [387, 299], [391, 293], [376, 261], [357, 261], [357, 288], [348, 291], [344, 304], [341, 320], [331, 327], [350, 327], [363, 326], [365, 322], [365, 321], [379, 295], [375, 281], [375, 276], [393, 282], [388, 263], [393, 267], [365, 304], [359, 304], [351, 316], [372, 316], [372, 306], [366, 288], [357, 341]]


In [19]:

def get_background_px(img_cube):

    hypercube = img_cube.load()
    display_band = hypercube[:, :, 50]

    background_coords = [] # List for background coordinates
    fig2, ax2 = plt.subplots()
    ax2.imshow(display_band, cmap='viridis')
    ax2.set_title("Click to select BACKGROUND pixels")

    def onclick_background(event):
        x, y = int(event.xdata), int(event.ydata)
        background_coords.append([y, x])
        ax2.plot(x, y, 'bo', markersize=5, markerfacecolor='none') # Blue circle marker
        fig2.canvas.draw()

    cid2 = fig2.canvas.mpl_connect('button_press_event', onclick_background)
    plt.show()

    return background_coords


In [20]:
backpx = get_background_px(img_cube1)


c:\ProgramData\anaconda3\envs\hyperspec\Lib\site-packages\spectral\io\spyfile.py:224: NaNValueWarning: Image data contains NaN values.
  warnings.warn('Image data contains NaN values.', NaNValueWarning)


In [22]:
print(backpx)

[[132, 95], [87, 68], [47, 46], [13, 24], [13, 52], [13, 93], [19, 164], [13, 232], [19, 311], [31, 417], [35, 503], [44, 571], [30, 616], [26, 574], [12, 515], [16, 455], [19, 391], [37, 371], [74, 275], [87, 129], [97, 68], [84, 35], [123, 47], [218, 75], [257, 54], [355, 44], [385, 43], [472, 39], [606, 42], [652, 42], [547, 34], [501, 32], [693, 20], [692, 78], [688, 116], [691, 140], [674, 154], [673, 182], [675, 199], [673, 235], [673, 290], [673, 322], [672, 340], [666, 383], [667, 413], [680, 416], [691, 350], [696, 280], [696, 274], [699, 263], [698, 257], [696, 194], [630, 121], [534, 108], [448, 95], [366, 157], [341, 65], [264, 80], [239, 130], [322, 182], [186, 25], [185, 73], [177, 123], [177, 180], [177, 192], [177, 215], [173, 249], [173, 293], [164, 390], [183, 427], [178, 336], [189, 379], [189, 457], [191, 501], [179, 598], [178, 611], [193, 526], [198, 568], [171, 555], [173, 469], [139, 410], [99, 332], [95, 224], [79, 183], [69, 155], [48, 103], [48, 82], [45, 164

In [42]:
img_cube1.asarray()

memmap([[[0.53088236, 0.475     , 0.48783782, ..., 0.11875   ,
          0.11014493, 0.11026785],
         [0.5029412 , 0.5277778 , 0.53918916, ..., 0.1180791 ,
          0.10935251, 0.11227273],
         [0.5135135 , 0.5277778 , 0.52500004, ..., 0.12344632,
          0.11014493, 0.12954545],
         ...,
         [0.5181818 , 0.58055556, 0.5428571 , ..., 0.1443038 ,
          0.1368    , 0.133     ],
         [0.5588235 , 0.53088236, 0.53088236, ..., 0.14339623,
          0.1281746 , 0.13168317],
         [0.6333333 , 0.57575756, 0.57      , ..., 0.14188312,
          0.1292    , 0.11399999]],

        [[0.53088236, 0.4486111 , 0.53918916, ..., 0.13494319,
          0.11702899, 0.1017857 ],
         [0.5588235 , 0.5013889 , 0.5905405 , ..., 0.12881356,
          0.11618704, 0.12090909],
         [0.5905405 , 0.5277778 , 0.55      , ..., 0.13418078,
          0.11014493, 0.11227273],
         ...,
         [0.5469697 , 0.5277778 , 0.5428571 , ..., 0.13829114,
          0.1292    , 0.1

In [43]:
img_cube_arr = img_cube1.asarray()
img_cube_arr

memmap([[[0.53088236, 0.475     , 0.48783782, ..., 0.11875   ,
          0.11014493, 0.11026785],
         [0.5029412 , 0.5277778 , 0.53918916, ..., 0.1180791 ,
          0.10935251, 0.11227273],
         [0.5135135 , 0.5277778 , 0.52500004, ..., 0.12344632,
          0.11014493, 0.12954545],
         ...,
         [0.5181818 , 0.58055556, 0.5428571 , ..., 0.1443038 ,
          0.1368    , 0.133     ],
         [0.5588235 , 0.53088236, 0.53088236, ..., 0.14339623,
          0.1281746 , 0.13168317],
         [0.6333333 , 0.57575756, 0.57      , ..., 0.14188312,
          0.1292    , 0.11399999]],

        [[0.53088236, 0.4486111 , 0.53918916, ..., 0.13494319,
          0.11702899, 0.1017857 ],
         [0.5588235 , 0.5013889 , 0.5905405 , ..., 0.12881356,
          0.11618704, 0.12090909],
         [0.5905405 , 0.5277778 , 0.55      , ..., 0.13418078,
          0.11014493, 0.11227273],
         ...,
         [0.5469697 , 0.5277778 , 0.5428571 , ..., 0.13829114,
          0.1292    , 0.1

In [45]:
img_cube_arr[0]

memmap([[0.53088236, 0.475     , 0.48783782, ..., 0.11875   , 0.11014493,
         0.11026785],
        [0.5029412 , 0.5277778 , 0.53918916, ..., 0.1180791 , 0.10935251,
         0.11227273],
        [0.5135135 , 0.5277778 , 0.52500004, ..., 0.12344632, 0.11014493,
         0.12954545],
        ...,
        [0.5181818 , 0.58055556, 0.5428571 , ..., 0.1443038 , 0.1368    ,
         0.133     ],
        [0.5588235 , 0.53088236, 0.53088236, ..., 0.14339623, 0.1281746 ,
         0.13168317],
        [0.6333333 , 0.57575756, 0.57      , ..., 0.14188312, 0.1292    ,
         0.11399999]], dtype=float32)

In [23]:

print(len(backpx))
print(len(plantsp))

259
44


In [24]:

def train_svm(image_cube, plant_coords, background_coords):

    hypercube = image_cube.load()
    # --- Build the training data from your selected points ---
    X_train = []
    y_train = []
    bands = hypercube.shape[2]

    # Add plant pixels (Class 1)
    for r, c in plant_coords:
        X_train.append(hypercube[r, c, :].reshape(bands))
        y_train.append(1)

    # Add background pixels (Class 0)
    for r, c in background_coords:
        X_train.append(hypercube[r, c, :].reshape(bands))
        y_train.append(0)

    X_train = np.array(X_train)
    y_train = np.array(y_train)

    print(f"Training data created successfully!")
    print(f"Plant samples: {len(plant_coords)}")
    print(f"Background samples: {len(background_coords)}")
    print(f"Total training samples: {X_train.shape[0]}")

    # --- Step 2: Train the SVM Classifier ---
    # The 'C' and 'gamma' parameters can be tuned, but defaults are a good start.
    from sklearn.svm import SVC

    classifier = SVC(C=550, gamma='scale')
    classifier.fit(X_train, y_train)
    print("Classifier trained successfully!")

    # --- Step 3: Predict on the Entire Image ---
    # Reshape the hypercube to a long list of pixels (n_pixels, n_bands)
    rows, cols, bands = hypercube.shape
    img_reshaped = hypercube.reshape(rows * cols, bands)

    # Predict the class for every pixel
    predicted_labels = classifier.predict(img_reshaped)

    # Reshape the prediction back into a 2D image mask
    final_mask = predicted_labels.reshape(rows, cols)

    # --- Visualize the Result ---
    plt.figure(figsize=(8, 8))
    plt.imshow(final_mask, cmap='gray')
    plt.title('Segmentation Mask from SVM Classifier')
    plt.show()

    return final_mask, classifier

def predict_svm(svm_classifier, image_cube):

    hypercube = image_cube.load()
    rows, cols, bands = image_cube.shape
    img_reshaped = hypercube.reshape(rows * cols, bands)

    # Predict the class for every pixel
    predicted_labels = svm_classifier.predict(img_reshaped)

    # Reshape the prediction back into a 2D image mask
    final_mask = predicted_labels.reshape(rows, cols)

    return final_mask


In [25]:

svm_mask, svm_classifier = train_svm(img_cube1, plantsp, backpx)

c:\ProgramData\anaconda3\envs\hyperspec\Lib\site-packages\spectral\io\spyfile.py:224: NaNValueWarning: Image data contains NaN values.
  warnings.warn('Image data contains NaN values.', NaNValueWarning)


Training data created successfully!
Plant samples: 44
Background samples: 259
Total training samples: 303
Classifier trained successfully!


ValueError: Input X contains NaN.
SVC does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [26]:
# %matplotlib qt
import numpy as np
from pathlib import Path
from utils.utils import run
from utils.train import SVMclassifier
from lima_beans.bean_utils import QtLinePicker, process_and_save_objects

In [27]:
X, y = run(img_cube1)


Captured 50 object pixels and 1 background pixels.


TypeError: '>=' not supported between instances of 'list' and 'int'

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

axes[0].imshow(svm_mask, cmap='gray')
axes[0].set_title('SVM')

axes[1].imshow(single_band, cmap='gray')
axes[1].set_title('Final Mask New')


### X

In [1]:
%matplotlib qt
import numpy as np
from pathlib import Path
from utils.utils import run
from utils.train import SVMclassifier
from lima_beans.bean_utils import QtLinePicker, process_and_save_objects

In [3]:
from biochar.biochar_utils import extract_reflectance_cube


In [ ]:
extract_reflectance_cube

import numpy as np
import os

def save_reflectance_np(filepath, ref):
    np.save(filepath, ref)
    

In [ ]:
data_root = 'C:/Users/baolab/Documents/Cubert/2025_09_03_11-12-59'
file_names = ['3_000_0012', '10_000_0010', '36_000_0008', '41_000_0014', '51_000_0008']

dest_root = 'data/numpy_files'
os.makedirs(dest_root, exist_ok=True)

for filename in file_names:

    cuvis_session_file = f'{data_root}/{filename}.cu3s'
    # print(cuvis_session_file)
    reflectance = extract_reflectance_cube(cuvis_session_file)

    dest_path = f'{dest_root}/{filename}.npy'
    # print(dest_path)
    save_reflectance_np(dest_path, reflectance)
